In [1]:
import xmlrpc.client

# =========================================================================
# 1. SERVER CONNECTION & HANDSHAKE
# =========================================================================
URL = "http://localhost:8069"
DB = "m.nushath"       
USERNAME = "user@company.com"  
API_KEY = "user"              

print("Connecting to Odoo API endpoints...")
common = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/common', allow_none=True)
models = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/object', allow_none=True)

uid = common.authenticate(DB, USERNAME, API_KEY, {})
if not uid:
    print("[CRITICAL] Authentication failed!")
    exit()

print(f"[SUCCESS] Connected securely as User ID: {uid}\n" + "="*70)


# =========================================================================
# 2. EXECUTE COMPANY RECORD PROVISIONING
# =========================================================================
print("[Step 1] Initializing New Company Creation Payload...")

new_company_vals = {
    'name': 'Soconi Global Trading Ltd - Test',
    'email': 'operations@soconi.global',
    'phone': '+1-800-555-0155',
    'street': '100 Innovation Way',
    'city': 'Tech City',
    'zip': '90210'
}

try:
    # Remember: passing a single dictionary inside a list returns a raw integer ID
    company_id = models.execute_kw(DB, uid, API_KEY, 'res.company', 'create', [new_company_vals])
    print(f" -> [SUCCESS] Company successfully generated in database! New Company ID: {company_id}")
    
    # =========================================================================
    # 3. VERIFY CREATION (READ BACK METADATA)
    # =========================================================================
    print("\n[Step 2] Verifying and reading back system metadata...")
    
    company_metadata = models.execute_kw(DB, uid, API_KEY, 'res.company', 'read', 
        [[company_id]], 
        {'fields': ['name', 'partner_id', 'currency_id']}
    )
    
    record = company_metadata[0]
    print(f"   [*] Registered Name: {record['name']}")
    print(f"   [*] Auto-Created Background Contact ID: {record['partner_id'][0]} ({record['partner_id'][1]})")
    print(f"   [*] Linked Base Currency: {record['currency_id'][1]}")

except Exception as e:
    print(f"[ERROR] Company provisioning aborted by server: {e}")

print("\n" + "="*70 + "\n[FINISHED] Company API creation demo sequence completed successfully.")

Connecting to Odoo API endpoints...
[SUCCESS] Connected securely as User ID: 5
[Step 1] Initializing New Company Creation Payload...
 -> [SUCCESS] Company successfully generated in database! New Company ID: 5

[Step 2] Verifying and reading back system metadata...
   [*] Registered Name: Soconi Global Trading Ltd - Test
   [*] Auto-Created Background Contact ID: 43 (Soconi Global Trading Ltd - Test)
   [*] Linked Base Currency: USD

[FINISHED] Company API creation demo sequence completed successfully.


In [2]:
# =========================================================================
# 2. DEFINE PROVISIONING TARGETS AND CONFIGURATION VARIABLES
# =========================================================================
# Target company matching your configuration screenshots
TARGET_COMPANY_ID = company_id 

# Define the exact company context parameters. This is the absolute secret
# to preventing internal validation errors when updating security matrices.
TARGET_COMPANY_CONTEXT = {
    'company_id': TARGET_COMPANY_ID,
    'allowed_company_ids': [TARGET_COMPANY_ID]
}


# =========================================================================
# 2. SCHEMA DISCOVERY (Self-Healing Field Name Finder)
# =========================================================================
print("[Step 1] Scanning database schema for the security field...")

user_schema = models.execute_kw(DB, uid, API_KEY, 'res.users', 'fields_get', 
    [], {'attributes': ['type', 'relation']}
)

dynamic_group_field = None
for field_name, properties in user_schema.items():
    if properties.get('type') == 'many2many' and properties.get('relation') == 'res.groups':
        dynamic_group_field = field_name
        break

if not dynamic_group_field:
    print("[CRITICAL ERROR] No relationship field linking users to groups exists!")
    exit()

print(f" -> Found target field: '{dynamic_group_field}'")


# =========================================================================
# 3. RESOLVE GROUP IDS
# =========================================================================
print("\n[Step 2] Resolving system database IDs for requested permissions...")

required_permissions = {
    'admin_access': ('base', 'group_erp_manager'),     
    'admin_system': ('base', 'group_system'),          # Flips the "Administrator" radio button
    'sales': ('sales_team', 'group_sale_salesman'),    
    'accounting': ('account', 'group_account_manager') 
}

group_db_ids = []
for access_key, (module, xml_name) in required_permissions.items():
    meta = models.execute_kw(DB, uid, API_KEY, 'ir.model.data', 'search_read', [
        [('module', '=', module), ('name', '=', xml_name)]
    ], {'fields': ['res_id'], 'limit': 1})
    if meta:
        group_db_ids.append(meta[0]['res_id'])

print(f" -> Resolved IDs: {group_db_ids}")


# =========================================================================
# 4. SINGLE-STEP CREATION
# =========================================================================
print(f"\n[Step 3] Firing atomic, single-step creation payload...")

NEW_LOGIN = 'jane.admin@soconi.global'
NEW_PASSWORD = 'SecurePassword123!'
    
creation_payload = {
    'name': 'Jane Doe (Executive Administrator)',
    'login': NEW_LOGIN,
    'email': 'jane.admin@soconi.global',
    'password': NEW_PASSWORD,
    
    # Base Multi-Company
    'company_id': TARGET_COMPANY_ID,                            
    'company_ids': [(6, 0, [TARGET_COMPANY_ID])],
    'share': False,
    
    # MAGIC HAPPENS HERE: Inject the permissions directly into creation!
    # Using (6, 0, [IDs]) replaces the default list with our exact permissions
    dynamic_group_field: [(6, 0, group_db_ids)]
}

try:
    # Pass the context directly into the create call to prevent company validation blocks
    new_user_id = models.execute_kw(DB, uid, API_KEY, 'res.users', 'create', 
        [creation_payload], 
        {'context': TARGET_COMPANY_CONTEXT}
    )
    print(f" -> [SUCCESS] User fully provisioned with roles in 1 step! ID: {new_user_id}")

except Exception as e:
    print(f"\n[CRITICAL ERROR] Creation failed: {e}")

print("\n" + "="*70 + "\n[FINISHED] Optimized single-step user script complete.")

[Step 1] Scanning database schema for the security field...
 -> Found target field: 'group_ids'

[Step 2] Resolving system database IDs for requested permissions...
 -> Resolved IDs: [2, 4, 28, 36]

[Step 3] Firing atomic, single-step creation payload...
 -> [SUCCESS] User fully provisioned with roles in 1 step! ID: 26

[FINISHED] Optimized single-step user script complete.


In [3]:
uid_new = common.authenticate(DB, NEW_LOGIN, NEW_PASSWORD, {})
if not uid_new:
    print("[CRITICAL] Authentication failed!")
    exit()

print(f"[SUCCESS] Connected securely as User ID: {uid_new}\n" + "="*70)


[SUCCESS] Connected securely as User ID: 26


In [16]:
# =========================================================================
# CREATE A GLOBAL DUMMY PRODUCT FOR ALL COMPANIES
# =========================================================================
print("[Step 1] Initializing Global Dummy Product for Accounting...")

# 1. Define the product parameters
product_payload = {
    'name': 'General Accounting Adjustment', 
    'default_code': 'DUMMY-GLOBAL',          
    'type': 'service',                       # Bypasses inventory tracking
    'list_price': 0.0,                       
    'standard_price': 0.0,                   
    'sale_ok': True,                         
    'purchase_ok': True,                     
    
    # THE SECRET TO GLOBAL RECORDS: 
    # Setting company_id to False makes it available to every company in the system.
    'company_id': False          
}

try:
    # 2. Execute creation (Context is still good to pass, though less strict for global records)
    DUMMY_PRODUCT_ID = models.execute_kw(DB, uid, API_KEY, 'product.template', 'create', 
        [product_payload],
        {'context': TARGET_COMPANY_CONTEXT}
    )
    print(f" -> [SUCCESS] Global Dummy Product created! Product Template ID: {DUMMY_PRODUCT_ID}")
    
    # =========================================================================
    # VERIFICATION
    # =========================================================================
    print("\n[VERIFICATION] Reading product metadata...")
    product_check = models.execute_kw(DB, uid, API_KEY, 'product.template', 'read', 
        [[DUMMY_PRODUCT_ID]], 
        {'fields': ['name', 'type', 'company_id']}
    )
    
    if product_check:
        item = product_check[0]
        # Odoo returns False for empty Many2one fields
        is_global = "Yes (Available to all)" if not item['company_id'] else f"No (Restricted to ID {item['company_id'][0]})"
        
        print(f"   [*] Product Name: {item['name']}")
        print(f"   [*] Product Type: {item['type'].upper()}")
        print(f"   [*] Is Global Product: {is_global}")

except Exception as e:
    print(f"\n[CRITICAL ERROR] Product creation failed: {e}")

[Step 1] Initializing Global Dummy Product for Accounting...
 -> [SUCCESS] Global Dummy Product created! Product Template ID: 15

[VERIFICATION] Reading product metadata...
   [*] Product Name: General Accounting Adjustment
   [*] Product Type: SERVICE
   [*] Is Global Product: Yes (Available to all)


In [5]:
# =========================================================================
# CREATE CORE CHART OF ACCOUNTS (COA) ENTRIES
# =========================================================================

# 1. ACCOUNTS RECEIVABLE (A/R) PAYLOAD
ar_payload = {
    'name': 'Accounts Receivable - PC1',
    'code': '120000',                  
    'account_type': 'asset_receivable',
    'reconcile': True,                 
    # V17+ ARCHITECTURE: Use plural company_ids mapping
    'company_ids': [(6, 0, [TARGET_COMPANY_ID])]
}

# 2. SALES INCOME PAYLOAD
sales_payload = {
    'name': 'Product Sales - PC1',
    'code': '400000',                  
    'account_type': 'income',          
    'reconcile': False,                
    # V17+ ARCHITECTURE: Use plural company_ids mapping
    'company_ids': [(6, 0, [TARGET_COMPANY_ID])]
}

try:
    # Execute creation pipelines
    AR_ACCOUNT_ID = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [ar_payload])
    print(f"[SUCCESS] Accounts Receivable created. ID: {AR_ACCOUNT_ID}")

    SALES_ACCOUNT_ID = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [sales_payload])
    print(f"[SUCCESS] Sales Income created. ID: {SALES_ACCOUNT_ID}")

except Exception as e:
    print(f"[CRITICAL ERROR] Account creation failed: {e}")

[SUCCESS] Accounts Receivable created. ID: 119
[SUCCESS] Sales Income created. ID: 120


In [6]:
# =========================================================================
# CREATE A SALES JOURNAL (CUSTOMER INVOICES)
# =========================================================================

print(f"[Step 1] Initializing Sales Journal for Company ID: {TARGET_COMPANY_ID}...")

journal_payload = {
    'name': 'Sales',
    'code': 'INV',                       # A short 3-5 character code used for sequence numbers (e.g., INV/2026/0001)
    'type': 'sale',                      # CRITICAL: Tells Odoo to use this for Customer Invoices
    'company_id': TARGET_COMPANY_ID,     # Journals are strictly bound to a single company
    'default_account_id': SALES_ACCOUNT_ID # Automatically routes revenue to your 400000 account
}

try:
    # Execute creation pipeline
    new_journal_id = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.journal', 'create', [journal_payload])
    print(f" -> [SUCCESS] Sales Journal created! Journal ID: {new_journal_id}")
    
    # =========================================================================
    # VERIFICATION
    # =========================================================================
    print("\n[VERIFICATION] Reading journal metadata...")
    journal_check = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.journal', 'read', 
        [[new_journal_id]], 
        {'fields': ['name', 'code', 'type', 'default_account_id']}
    )
    
    if journal_check:
        item = journal_check[0]
        linked_account = item['default_account_id'][1] if item['default_account_id'] else "None"
        
        print(f"   [*] Journal Name: {item['name']} ({item['code']})")
        print(f"   [*] Engine Type: {item['type'].upper()}")
        print(f"   [*] Default Income Account: {linked_account}")

except Exception as e:
    print(f"\n[CRITICAL ERROR] Journal creation failed: {e}")

[Step 1] Initializing Sales Journal for Company ID: 5...
 -> [SUCCESS] Sales Journal created! Journal ID: 42

[VERIFICATION] Reading journal metadata...
   [*] Journal Name: Sales (INV)
   [*] Engine Type: SALE
   [*] Default Income Account: 400000 Product Sales - PC1


In [8]:
# =========================================================================
# CREATE BANK RECONCILIATION ACCOUNTS (MODERN ODOO 17+)
# =========================================================================

# 1. MAIN BANK ACCOUNT PAYLOAD
# Tracks your actual cleared cash balance
main_bank_payload = {
    'name': 'Main Bank Account - PC1',
    'code': '101400',                  
    'account_type': 'asset_cash',      # CRITICAL: Identifies this as liquid cash
    'reconcile': False,                # Bank asset accounts are not reconciled line-by-line
    'company_ids': [(6, 0, [TARGET_COMPANY_ID])]
}

# 2. OUTSTANDING RECEIPTS PAYLOAD
# Holds money registered in Odoo but not yet cleared by the bank
outstanding_receipts_payload = {
    'name': 'Outstanding Receipts - PC1',
    'code': '101401',                  
    'account_type': 'asset_current',   # Standard current asset
    'reconcile': True,                 # CRITICAL: Must be True to match payments to statements
    'company_ids': [(6, 0, [TARGET_COMPANY_ID])]
}

# 3. BANK SUSPENSE PAYLOAD
# Holds mystery bank statement lines until you match them to an invoice
suspense_payload = {
    'name': 'Bank Suspense Account - PC1',
    'code': '101402',                  
    'account_type': 'asset_current',   
    'reconcile': True,                # Cleared automatically by Odoo's reconciliation widget
    'company_ids': [(6, 0, [TARGET_COMPANY_ID])]
}

try:
    # Execute creation pipelines
    MAIN_BANK_ACCOUNT_ID = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [main_bank_payload])
    print(f"[SUCCESS] Main Bank Account created. ID: {MAIN_BANK_ACCOUNT_ID}")

    OUTSTANDING_RECEIPTS_ACCOUNT_ID = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [outstanding_receipts_payload])
    print(f"[SUCCESS] Outstanding Receipts created. ID: {OUTSTANDING_RECEIPTS_ACCOUNT_ID}")
    
    SUSPENSE_ACCOUNT_ID = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [suspense_payload])
    print(f"[SUCCESS] Bank Suspense created. ID: {SUSPENSE_ACCOUNT_ID}")
    
except Exception as e:
    print(f"[CRITICAL ERROR] Account creation failed: {e}")

[SUCCESS] Main Bank Account created. ID: 124
[SUCCESS] Outstanding Receipts created. ID: 125
[SUCCESS] Bank Suspense created. ID: 126


In [10]:
# =========================================================================
# 3. PHASE 1: CREATE THE BANK JOURNAL
# =========================================================================
print(f"\n[Step 1] Creating Bank Journal for Company ID: {TARGET_COMPANY_ID}...")

journal_payload = {
    'name': 'Bank',
    'code': 'BNK',                       
    'type': 'bank',                      
    'company_id': TARGET_COMPANY_ID,     
    'default_account_id': MAIN_BANK_ACCOUNT_ID,  
    'suspense_account_id': SUSPENSE_ACCOUNT_ID   
}

try:
    # Execute journal creation
    new_journal_id = models.execute_kw(DB, uid, API_KEY, 'account.journal', 'create', [journal_payload])
    print(f" -> [SUCCESS] Bank Journal created! Journal ID: {new_journal_id}")
    
    # =========================================================================
    # 4. PHASE 2: LINK OUTSTANDING RECEIPTS TO THE AUTO-GENERATED PAYMENT LINE
    # =========================================================================
    print(f"\n[Step 2] Locating auto-generated inbound payment lines for Journal {new_journal_id}...")
    
    # Search for the Inbound Payment Method Line Odoo just created for this specific journal
    inbound_lines = models.execute_kw(DB, uid, API_KEY, 'account.payment.method.line', 'search_read', 
        [[
            ('journal_id', '=', new_journal_id), 
            ('payment_type', '=', 'inbound')
        ]], 
        {'fields': ['id', 'name']}
    )
    
    if not inbound_lines:
        print(" -> [WARNING] No inbound payment lines were auto-generated by the engine.")
    else:
        line_ids = [line['id'] for line in inbound_lines]
        
        # Write the Outstanding Receipts account ID directly to those lines
        models.execute_kw(DB, uid, API_KEY, 'account.payment.method.line', 'write', [
            line_ids, 
            {'payment_account_id': OUTSTANDING_RECEIPTS_ACCOUNT_ID}
        ])
        print(f" -> [SUCCESS] Outstanding Receipts mapped to Inbound Payment Line(s): {line_ids}")

    # =========================================================================
    # 5. VERIFICATION
    # =========================================================================
    print("\n" + "="*70)
    print(f"[VERIFICATION] Confirming complete journal architecture in database...")
    print("="*70)
    
    # Verify Journal
    journal_check = models.execute_kw(DB, uid, API_KEY, 'account.journal', 'read', 
        [[new_journal_id]], {'fields': ['name', 'default_account_id', 'suspense_account_id']}
    )
    j_data = journal_check[0]
    print(f"   [*] Journal Name: {j_data['name']}")
    print(f"   [*] Main Account: {j_data['default_account_id'][1] if j_data['default_account_id'] else 'None'}")
    print(f"   [*] Suspense Account: {j_data.get('suspense_account_id')[1] if j_data.get('suspense_account_id') else 'None'}")
    
    # Verify Payment Lines
    if inbound_lines:
        verify_lines = models.execute_kw(DB, uid, API_KEY, 'account.payment.method.line', 'read', 
            [line_ids], {'fields': ['name', 'payment_account_id']}
        )
        for line in verify_lines:
            linked_acc = line.get('payment_account_id')
            acc_name = linked_acc[1] if linked_acc else "UNASSIGNED"
            print(f"   [*] Inbound Method '{line['name']}' ➔ Routes to: {acc_name}")

except Exception as e:
    print(f"\n[CRITICAL ERROR] Execution pipeline halted: {e}")

print("\n" + "="*70 + "\n[FINISHED] Complete Bank Journal provisioning script executed.")


[Step 1] Creating Bank Journal for Company ID: 5...
 -> [SUCCESS] Bank Journal created! Journal ID: 44

[Step 2] Locating auto-generated inbound payment lines for Journal 44...
 -> [SUCCESS] Outstanding Receipts mapped to Inbound Payment Line(s): [48]

[VERIFICATION] Confirming complete journal architecture in database...
   [*] Journal Name: Bank
   [*] Main Account: Main Bank Account - PC1
   [*] Suspense Account: Bank Suspense Account - PC1
   [*] Inbound Method 'Manual Payment' ➔ Routes to: Outstanding Receipts - PC1

[FINISHED] Complete Bank Journal provisioning script executed.


In [14]:
# =========================================================================
# PHASE 1: CREATE THE ACTIVE GUEST PROFILE
# =========================================================================
print("\n[Step 1] Creating temporary active guest profile...")

incoming_order_data = {
    'name': 'Alex Guest',
    'email': 'alex.guest@example.com',
    'amount': 199.99,
    'quantity': 1.0
}

partner_payload = {
    'name': incoming_order_data['name'],
    'email': incoming_order_data['email'],
    'active': False,  # MUST be active during invoice creation
    'company_id': TARGET_COMPANY_ID,     
    'comment': 'Auto-generated Guest Checkout' 
}

CUSTOMER_PARTNER_ID = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'res.partner', 'create', [partner_payload])
print(f" -> [CREATED] Active guest profile generated. Partner ID: {CUSTOMER_PARTNER_ID}")


[Step 1] Creating temporary active guest profile...
 -> [CREATED] Active guest profile generated. Partner ID: 47


In [15]:
# =========================================================================
# 3. DYNAMICALLY FETCH BANK JOURNAL (Borrowed from your old code!)
# =========================================================================
print("\n[Step 1] Dynamically locating the default Bank Journal...")

bank_journals = models.execute_kw(DB, uid, API_KEY, 'account.journal', 'search_read', [
    [('type', '=', 'bank'), ('company_id', '=', TARGET_COMPANY_ID)]
], {'fields': ['id', 'name'], 'limit': 1})

if not bank_journals:
    print("[CRITICAL ERROR] No Bank journal found for this company!")
    exit()

BANK_JOURNAL_ID = bank_journals[0]['id']
print(f" -> Found Bank Journal: '{bank_journals[0]['name']}' (ID: {BANK_JOURNAL_ID})")


[Step 1] Dynamically locating the default Bank Journal...
 -> Found Bank Journal: 'Bank' (ID: 44)


In [18]:
# =========================================================================
# 4. CREATE & POST THE CUSTOMER INVOICE (No Sales Journal ID required)
# =========================================================================
from datetime import date


print(f"\n[Step 2] Generating Customer Invoice...")

# Example external order number from your frontend/website
external_order_id = "WEB-ORD-88219" 

invoice_payload = {
    'move_type': 'out_invoice',          
    'partner_id': CUSTOMER_PARTNER_ID,
    'company_id': TARGET_COMPANY_ID,
    
    # ADD THIS: This populates the "Reference" column on the Sales Journal Entry
    'ref': external_order_id,
    
    # Optional but recommended: Sets the payment communication memo
    'payment_reference': external_order_id, 
    
    'invoice_line_ids': [(0, 0, {
        'product_id': DUMMY_PRODUCT_ID,
        'quantity': 1.0,
        'price_unit': incoming_order_data['amount'],
        
        # ADD THIS: Gives the specific journal line item a clean description 
        # instead of just defaulting to the Dummy Product's name
        'name': f'Checkout Payment for {external_order_id}' 
    })]
}

try:
    invoice_id = models.execute_kw(DB, uid, API_KEY, 'account.move', 'create', 
        [invoice_payload], {'context': TARGET_COMPANY_CONTEXT}
    )
    models.execute_kw(DB, uid, API_KEY, 'account.move', 'action_post', 
        [[invoice_id]], {'context': TARGET_COMPANY_CONTEXT}
    )
    print(f" -> [SUCCESS] Invoice Posted! (Sales Journal auto-assigned by Odoo)")

    # =========================================================================
    # 5. REGISTER THE PAYMENT
    # =========================================================================
    print(f"\n[Step 3] Processing automatic bank payment...")
    
    wizard_context = {
        'company_id': TARGET_COMPANY_ID,
        'allowed_company_ids': [TARGET_COMPANY_ID],
        'active_model': 'account.move',
        'active_ids': [invoice_id]
    }
    
    payment_wizard_payload = {
        'journal_id': BANK_JOURNAL_ID, # Passed dynamically!
        'amount': incoming_order_data['amount'],
        'payment_date': str(date.today())
    }
    
    wizard_id = models.execute_kw(DB, uid, API_KEY, 'account.payment.register', 'create', 
        [payment_wizard_payload], {'context': wizard_context}
    )
    
    models.execute_kw(DB, uid, API_KEY, 'account.payment.register', 'action_create_payments', 
        [[wizard_id]], {'context': wizard_context}
    )
    print(" -> [SUCCESS] Payment Registered! Invoice fully paid.")

except Exception as e:
    print(f"\n[CRITICAL ERROR] Execution pipeline halted: {e}")


[Step 2] Generating Customer Invoice...
 -> [SUCCESS] Invoice Posted! (Sales Journal auto-assigned by Odoo)

[Step 3] Processing automatic bank payment...
 -> [SUCCESS] Payment Registered! Invoice fully paid.
